# Lab 11 확장: APIM = Microsoft Graph 감사 게이트웨이 — 로깅 데모

이 노트북은 **APIM 을 여러 솔루션의 Graph 게이트웨이로 쓸 때, "누가 · 무엇을 · 어떻게 · 결과"를 감사(audit) 수준으로 남길 수 있음**을 직접 보여준다.
관련 리포트: [`docs/graph-gateway-audit-logging.md`](../../docs/graph-gateway-audit-logging.md) · 정책: [`policies/graph-audit-policy.xml`](../../policies/graph-audit-policy.xml)

## 두 가지 옵션
| | Option A — `trace`(App Insights) | Option B — Event Hub(→Capture→Blob) |
|---|---|---|
| 남기는 것 | 감사 **메타데이터** (누가/무엇/어떻게/결과/차단사유) | 위 + **전문(full body) 무손실** 원장 |
| 저장소 | App Insights `traces` | Event Hub → (Capture) Blob/ADLS |
| 상한 | 메시지·차원 크기 제한(수 KB), 샘플링·일일한도 영향 | 이벤트 최대 ~1MB, 샘플링 없음, 접근 분리 |
| 비용/구성 | 이미 배선됨(추가 0) | Event Hub(+Capture) 필요 |
| 데모 | **즉시 실행·조회** | 옵트인 프로비저닝 후 라이브 소비 |

> **레이턴시 오해 정정** — 두 경로 모두 **비동기 버퍼링**이라 요청 지연과 무관하다. Event Hub 를 쓰는 이유는 *레이턴시*가 아니라 **용량(크기 상한·샘플링 회피)과 무손실·불변 아카이브** 때문. (리포트 §3·§6·§7)

## Event Hub 를 처음 쓴다면 (30초)
- Event Hub = 이벤트 **버스/큐**. APIM(`log-to-eventhub`)이 감사 레코드를 던지고, 여러 소비자가 각자 읽는다(consumer group).
- 자체 보존은 1~7일(임시). **영구 저장은 Event Hubs Capture 가 Blob 에 Avro 로 자동 기록** → 최종 저장소는 Blob.
- "그냥 Blob 에 쌓으면?" → APIM 정책엔 `log-to-blob` 이 **없다.** 커스텀 감사 payload 의 유일한 네이티브 출구가 Event Hub. (리포트 §7)

## 실행 순서
1. 셀 1~2 — 설정 · 감사 조각 정의
2. 셀 3~4 — **BEFORE** (정책 전 로깅)
3. 셀 5~8 — **Option A** 적용 → 트래픽 → `traces` 조회
4. 셀 9~12 — **Option B**(옵트인) 프로비저닝 → 적용 → 라이브 소비
5. 셀 13 — 원복

> ⚠️ App Insights 수집은 **2~5분 지연**될 수 있다 — 조회 셀은 잠시 후 재실행하면 채워진다.
> ⚠️ 사전조건: `az login`, `.env` 의 `APIM_KEY_GRAPH_USERS`(+선택 MAIL/SHAREPOINT), 그리고 Lab 11 의 Graph API/정책이 이미 배포돼 있어야 한다.

In [ ]:
# 셀 1: 환경 + 헬퍼 + 리소스 자동 탐색
import os, sys, json, time, subprocess, tempfile
from pathlib import Path
import requests

def load_env(path=".env"):
    env = {}; p = Path(path)
    if not p.exists(): p = Path("../../.env")
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line: continue
            k, v = line.split("=", 1); env[k.strip()] = v.strip().strip('"').strip("'")
    return env
env = load_env()

APIM_URL = env.get("APIM_URL", "").rstrip("/")
KEY_USERS = env.get("APIM_KEY_GRAPH_USERS", ""); KEY_MAIL = env.get("APIM_KEY_GRAPH_MAIL", "")
KEY_SP = env.get("APIM_KEY_GRAPH_SHAREPOINT", ""); TEST_USER_ID = env.get("GRAPH_TEST_USER_ID", "")
if TEST_USER_ID.startswith("<"): TEST_USER_ID = ""
GRAPH_BASE = f"{APIM_URL}/graph"

def az(args):
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode
def az_json(args):
    out, err, rc = az(args)
    if rc != 0 or not out: return None
    try: return json.loads(out)
    except json.JSONDecodeError: return out

SUBSCRIPTION_ID = env.get("AZURE_SUBSCRIPTION_ID") or (az_json(["account","show","--query","id","-o","json"]) or "")
RESOURCE_GROUP = env.get("RESOURCE_GROUP",""); APIM_NAME = env.get("APIM_NAME","")
if not (RESOURCE_GROUP and APIM_NAME):
    apims = az_json(["apim","list","--query","[].{name:name,rg:resourceGroup}","-o","json"]) or []
    if apims:
        APIM_NAME = APIM_NAME or apims[0]["name"]; RESOURCE_GROUP = RESOURCE_GROUP or apims[0]["rg"]
assert SUBSCRIPTION_ID and APIM_NAME and RESOURCE_GROUP, "az login / APIM 배포를 확인하세요."
ARM_API = "2024-06-01-preview"
ARM_BASE = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
            f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}")

def arm(method, path, body=None, query=None, api=ARM_API):
    url = f"{ARM_BASE}{path}?api-version={api}"
    args = ["rest","--method",method,"--url",url]
    if query: args += ["--query",query,"-o","json"]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w",suffix=".json",delete=False); json.dump(body,tmp); tmp.close()
        args += ["--headers","Content-Type=application/json","--body",f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp: os.unlink(tmp.name)
    return out, err, rc
def arm_q(path, jmes, api=ARM_API):
    url = f"{ARM_BASE}{path}?api-version={api}"
    out, err, rc = az(["rest","--method","GET","--url",url,"--query",jmes,"-o","tsv"])
    return out.strip() if rc == 0 else ""

# App Insights App ID (KQL 조회용)
APP_INSIGHTS_APP_ID = (env.get("APP_INSIGHTS_APP_ID","") or "").strip().strip('"')
if APP_INSIGHTS_APP_ID.startswith("<"): APP_INSIGHTS_APP_ID = ""
if not APP_INSIGHTS_APP_ID:
    comp = az_json(["monitor","app-insights","component","show","-g",RESOURCE_GROUP,"--query","[0].appId","-o","json"]) or ""
    APP_INSIGHTS_APP_ID = comp if isinstance(comp,str) else ""

def _ai_token():
    out, err, rc = az(["account","get-access-token","--resource","https://api.applicationinsights.io","--query","accessToken","-o","tsv"])
    return out.strip() if rc == 0 else ""
def query_ai(kql):
    if not APP_INSIGHTS_APP_ID: print("  ⚠️ APP_INSIGHTS_APP_ID 미설정 → KQL 건너뜀"); return None
    tok = _ai_token()
    if not tok: print("  ⚠️ 토큰 획득 실패"); return None
    r = requests.post(f"https://api.applicationinsights.io/v1/apps/{APP_INSIGHTS_APP_ID}/query",
                      headers={"Authorization":f"Bearer {tok}","Content-Type":"application/json"},
                      json={"query":kql}, timeout=60)
    if r.status_code != 200: print(f"  ⚠️ KQL HTTP {r.status_code}: {r.text[:160]}"); return None
    return r.json()
def show(result):
    if not result or not result.get("tables") or not result["tables"][0]["rows"]:
        print("  (결과 없음 — App Insights 수집 지연 2~5분일 수 있음. 잠시 후 이 셀을 다시 실행하세요.)"); return 0
    t = result["tables"][0]; cols = [c["name"] for c in t["columns"]]; rows = t["rows"]
    w = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i,c in enumerate(cols)]
    print("  " + "  ".join(f"{c:<{w[i]}}" for i,c in enumerate(cols)))
    print("  " + "  ".join("-"*w[i] for i in range(len(cols))))
    for r in rows: print("  " + "  ".join(f"{str(v):<{w[i]}}" for i,v in enumerate(r)))
    return len(rows)

# Graph API 리소스 id 탐색 (path=='graph')
GRAPH_API_ID = env.get("GRAPH_API_ID","") or arm_q("/apis","value[?properties.path=='graph'].name | [0]")
if not GRAPH_API_ID or GRAPH_API_ID == "null":
    GRAPH_API_ID = arm_q("/apis","value[?contains(properties.path,'graph')].name | [0]")
assert GRAPH_API_ID and GRAPH_API_ID != "null", "path='graph' API를 찾지 못함 — Lab 11에서 Graph API를 먼저 추가하세요."

def get_policy(api_id):
    url = f"{ARM_BASE}/apis/{api_id}/policies/policy?api-version={ARM_API}&format=rawxml"
    out, err, rc = az(["rest","--method","GET","--url",url])
    if rc != 0 or not out: return ""
    try: return json.loads(out)["properties"]["value"]
    except Exception: return out
def put_policy(api_id, xml):
    return arm("PUT", f"/apis/{api_id}/policies/policy", body={"properties":{"format":"rawxml","value":xml}})

def graph_get(path, sub_key, params=None):
    r = requests.get(f"{GRAPH_BASE}{path}", headers={"Ocp-Apim-Subscription-Key": sub_key}, params=params, timeout=30)
    print(f"  GET {path}  → HTTP {r.status_code}"); return r

def ensure_ai_verbosity(level="information"):
    """trace→App Insights 방출 조건은 '정책 severity ≥ 진단 verbosity'. 진단 verbosity 기본값은 사실상 error 라
    severity=information 트레이스가 전부 드롭된다(=traces 0건의 1차 원인). 전역(서비스) 대신 graph API-레벨
    진단에만 information 을 적용한다(다른 API 영향 0). 이 진단이 원래 있었는지(preexisted)를 반환한다."""
    logger_id = arm_q("/diagnostics/applicationinsights", "properties.loggerId") or \
                arm_q("/loggers", "value[?properties.loggerType=='applicationInsights'].id | [0]")
    if not logger_id:
        print("  \u26a0\ufe0f App Insights 로거를 찾지 못함 — trace 조회가 비활성일 수 있음"); return None
    path = f"/apis/{GRAPH_API_ID}/diagnostics/applicationinsights"
    preexisted = bool(arm_q(path, "name"))
    body = {"properties": {"loggerId": logger_id, "verbosity": level,
            "sampling": {"samplingType": "fixed", "percentage": 100.0},
            "logClientIp": True, "httpCorrelationProtocol": "W3C"}}
    _o, _e, rc = arm("PUT", path, body=body)
    print(f"  \u2705 API-레벨 App Insights 진단 verbosity={level} (전역 미변경)" if rc == 0 else f"  \u26a0\ufe0f 진단 설정 실패: {_e[:160]}")
    return preexisted

def wait_propagation(sec=150):
    """\u26a0\ufe0f APIM 은 정책·진단 변경을 게이트웨이에 전파하는 데 ~1~2분 걸린다. 전파 전 트래픽은 '구 설정'으로
    처리돼 로그가 누락된다(=traces 0건의 2차 원인). 그래서 변경 직후 반드시 대기한 뒤 트래픽을 보낸다."""
    print(f"  \u23f3 설정 전파 대기 {sec}초 (이 대기를 건너뛰면 traces 가 0건으로 보인다) …")
    time.sleep(sec)
    print("  \u2705 전파 대기 완료")


# --- Option C 헬퍼: 진단설정(GatewayLogs) → 같은 graph-audit Event Hub 비교용 ---------
APIM_RESOURCE_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"

def eh_ns_name():
    import hashlib
    return f"ehnsgraphaudit{hashlib.sha1(RESOURCE_GROUP.encode()).hexdigest()[:8]}"

def ns_rule_id(ns, rule="RootManageSharedAccessKey"):
    rid = az_json(["eventhubs","namespace","authorization-rule","show","-g",RESOURCE_GROUP,
        "--namespace-name",ns,"-n",rule,"--query","id","-o","json"])
    if not rid:
        az(["eventhubs","namespace","authorization-rule","create","-g",RESOURCE_GROUP,
            "--namespace-name",ns,"-n",rule,"--rights","Manage","Send","Listen"])
        rid = az_json(["eventhubs","namespace","authorization-rule","show","-g",RESOURCE_GROUP,
            "--namespace-name",ns,"-n",rule,"--query","id","-o","json"])
    return rid or ""

def enable_diag_to_eventhub(name="graph-audit-diag"):
    ns = eh_ns_name(); rid = ns_rule_id(ns)
    if not rid: return False, "네임스페이스 권한 규칙 없음"
    logs = json.dumps([{"category":"GatewayLogs","enabled":True}])
    o,e,rc = az(["monitor","diagnostic-settings","create","--name",name,"--resource",APIM_RESOURCE_ID,
        "--event-hub-rule",rid,"--event-hub","graph-audit","--logs",logs])
    return rc==0, (e or o)

def disable_diag_to_eventhub(name="graph-audit-diag"):
    o,e,rc = az(["monitor","diagnostic-settings","delete","--name",name,"--resource",APIM_RESOURCE_ID])
    return rc==0

print("✅ 설정 완료")
print("  APIM:", APIM_NAME, "| RG:", RESOURCE_GROUP)
print("  Graph API id:", GRAPH_API_ID, "| base:", GRAPH_BASE)
print("  App Insights App ID:", (APP_INSIGHTS_APP_ID[:8]+"...") if APP_INSIGHTS_APP_ID else "(미설정 — KQL 비활성)")

In [ ]:
# 셀 2: 감사 정책 조각 정의 (inbound 변수 → trace / log-to-eventhub 주입용)
# ⚠️ 속성값(value="...") 안의 따옴표는 ARM rawxml 규칙상 &quot; 로 이스케이프한다.

SETVARS = r"""
        <set-variable name="auditProduct" value="@(context.Product?.Name ?? &quot;-&quot;)" />
        <set-variable name="auditSubName" value="@(context.Subscription?.Name ?? &quot;-&quot;)" />
        <set-variable name="auditPath" value="@(context.Request.OriginalUrl.Path)" />
        <set-variable name="auditTargetUser" value="@(context.Request.MatchedParameters.ContainsKey(&quot;userId&quot;) ? context.Request.MatchedParameters[&quot;userId&quot;] : (context.Request.MatchedParameters.ContainsKey(&quot;user-id&quot;) ? context.Request.MatchedParameters[&quot;user-id&quot;] : &quot;-&quot;))" />
        <set-variable name="auditSelect" value="@(context.Request.OriginalUrl.Query.ContainsKey(&quot;$select&quot;) ? string.Join(&quot;,&quot;, context.Request.OriginalUrl.Query[&quot;$select&quot;]) : &quot;-&quot;)" />
        <set-variable name="auditFilter" value="@(context.Request.OriginalUrl.Query.ContainsKey(&quot;$filter&quot;) ? string.Join(&quot;,&quot;, context.Request.OriginalUrl.Query[&quot;$filter&quot;]) : &quot;-&quot;)" />
        <set-variable name="auditKeyHint" value="@{ var k = context.Request.Headers.GetValueOrDefault(&quot;Ocp-Apim-Subscription-Key&quot;, &quot;&quot;); return string.IsNullOrEmpty(k) ? &quot;-&quot; : (k.Length > 4 ? k.Substring(0, 4) + &quot;****&quot; : &quot;****&quot;); }" />
        <set-variable name="auditIdentity" value="system-assigned" />"""

INBOUND_TRACE = r"""
        <trace source="graph-audit" severity="information">
            <message>@(&quot;GRAPH-AUDIT-REQ &quot; + context.RequestId.ToString())</message>
            <metadata name="phase" value="inbound" />
            <metadata name="product" value="@((string)context.Variables[&quot;auditProduct&quot;])" />
            <metadata name="subName" value="@((string)context.Variables[&quot;auditSubName&quot;])" />
            <metadata name="path" value="@((string)context.Variables[&quot;auditPath&quot;])" />
            <metadata name="targetUser" value="@((string)context.Variables[&quot;auditTargetUser&quot;])" />
            <metadata name="select" value="@((string)context.Variables[&quot;auditSelect&quot;])" />
            <metadata name="filter" value="@((string)context.Variables[&quot;auditFilter&quot;])" />
            <metadata name="keyHint" value="@((string)context.Variables[&quot;auditKeyHint&quot;])" />
            <metadata name="identity" value="@((string)context.Variables[&quot;auditIdentity&quot;])" />
            <metadata name="callerIp" value="@(context.Request.IpAddress)" />
        </trace>"""

OUTBOUND_TRACE = r"""
        <trace source="graph-audit" severity="information">
            <message>@(&quot;GRAPH-AUDIT-RES &quot; + context.RequestId.ToString())</message>
            <metadata name="phase" value="outbound" />
            <metadata name="product" value="@((string)context.Variables[&quot;auditProduct&quot;])" />
            <metadata name="subName" value="@((string)context.Variables[&quot;auditSubName&quot;])" />
            <metadata name="path" value="@((string)context.Variables[&quot;auditPath&quot;])" />
            <metadata name="targetUser" value="@((string)context.Variables[&quot;auditTargetUser&quot;])" />
            <metadata name="select" value="@((string)context.Variables[&quot;auditSelect&quot;])" />
            <metadata name="identity" value="@((string)context.Variables[&quot;auditIdentity&quot;])" />
            <metadata name="backendUrl" value="@(context.Request.Url.ToString())" />
            <metadata name="responseCode" value="@(context.Response.StatusCode.ToString())" />
        </trace>"""

ONERROR_TRACE = r"""
        <trace source="graph-audit" severity="information">
            <message>@(&quot;GRAPH-AUDIT-ERR &quot; + context.RequestId.ToString())</message>
            <metadata name="phase" value="on-error" />
            <metadata name="product" value="@((string)context.Variables[&quot;auditProduct&quot;])" />
            <metadata name="path" value="@((string)context.Variables[&quot;auditPath&quot;])" />
            <metadata name="targetUser" value="@((string)context.Variables[&quot;auditTargetUser&quot;])" />
            <metadata name="status" value="@(context.Response != null ? context.Response.StatusCode.ToString() : &quot;-&quot;)" />
            <metadata name="lastError" value="@(context.LastError?.Message ?? &quot;-&quot;)" />
        </trace>"""

# log-to-eventhub 는 '요소 텍스트' 컨텍스트 → 내부 큰따옴표는 그대로 두어도 XML 유효(<, & 만 회피).
EH_OUT = r"""
        <log-to-eventhub logger-id="graph-audit-eventhub">@{
            var body = new Newtonsoft.Json.Linq.JObject();
            body["ts"] = DateTime.UtcNow.ToString("o");
            body["requestId"] = context.RequestId.ToString();
            body["phase"] = "outbound";
            body["product"] = (string)context.Variables["auditProduct"];
            body["subName"] = (string)context.Variables["auditSubName"];
            body["path"] = (string)context.Variables["auditPath"];
            body["targetUser"] = (string)context.Variables["auditTargetUser"];
            body["select"] = (string)context.Variables["auditSelect"];
            body["filter"] = (string)context.Variables["auditFilter"];
            body["keyHint"] = (string)context.Variables["auditKeyHint"];
            body["identity"] = (string)context.Variables["auditIdentity"];
            body["callerIp"] = context.Request.IpAddress;
            body["backendUrl"] = context.Request.Url.ToString();
            body["responseCode"] = context.Response.StatusCode;
            return body.ToString(Newtonsoft.Json.Formatting.None);
        }</log-to-eventhub>"""

EH_ERR = r"""
        <log-to-eventhub logger-id="graph-audit-eventhub">@{
            var body = new Newtonsoft.Json.Linq.JObject();
            body["ts"] = DateTime.UtcNow.ToString("o");
            body["requestId"] = context.RequestId.ToString();
            body["phase"] = "on-error";
            body["product"] = (string)context.Variables["auditProduct"];
            body["path"] = (string)context.Variables["auditPath"];
            body["targetUser"] = (string)context.Variables["auditTargetUser"];
            body["identity"] = (string)context.Variables["auditIdentity"];
            body["status"] = context.Response != null ? context.Response.StatusCode : 0;
            body["lastError"] = context.LastError != null ? context.LastError.Message : "-";
            return body.ToString(Newtonsoft.Json.Formatting.None);
        }</log-to-eventhub>"""

def inject(xml, inbound_extra, outbound_extra, onerror_extra):
    """기존 정책의 게이트/MI 인증은 보존하고 감사 조각만 끼워넣는다."""
    assert "<inbound>" in xml, "정책에 <inbound> 가 없습니다."
    xml = xml.replace("<inbound>", "<inbound>" + inbound_extra, 1)
    if "</outbound>" in xml:
        xml = xml.replace("</outbound>", outbound_extra + "\n    </outbound>", 1)
    else:
        xml = xml.replace("</policies>", "<outbound><base />" + outbound_extra + "</outbound></policies>", 1)
    if "</on-error>" in xml:
        xml = xml.replace("</on-error>", onerror_extra + "\n    </on-error>", 1)
    else:
        xml = xml.replace("</policies>", "<on-error><base />" + onerror_extra + "</on-error></policies>", 1)
    return xml

print("✅ 감사 조각 준비 완료 (SETVARS, *_TRACE, EH_*, inject())")

## BEFORE — 정책을 넣기 전, 지금 로깅으로 보이는 것

Graph 호출은 서비스 수준 진단으로 `requests`(App Insights) / `ApiManagementGatewayLogs`(Log Analytics)에 남는다.
`url`·`resultCode`·`BackendUrl` 은 보이지만, **누구의 어떤 사용자 데이터를, 어떤 신원으로 가져갔는지**는 필드로 없다.

In [ ]:
# 셀 4: BEFORE — 소량 트래픽 후 App Insights 'requests' 조회
if KEY_USERS and "<" not in KEY_USERS:
    graph_get("/users", KEY_USERS, params={"$select":"displayName,mail","$top":"3"})
else:
    print("  ⚠️ APIM_KEY_GRAPH_USERS 미설정 — 트래픽 생략(기존 로그로 조회)")

print("\n▶ App Insights 'requests' (수집 지연 시 잠시 후 재실행)")
show(query_ai("""
requests
| where timestamp > ago(30m)
| where url has "/graph"
| project timestamp, name, resultCode,
    targetUserDim = iif(isempty(tostring(customDimensions.targetUser)), "(없음)", tostring(customDimensions.targetUser)),
    identityDim   = iif(isempty(tostring(customDimensions.identity)),   "(없음)", tostring(customDimensions.identity))
| order by timestamp desc
| take 10
"""))
print("\n해석: url·resultCode 는 남지만 targetUser/identity 같은 감사 필드는 '(없음)'. → 정책으로 채운다.")

## Option A — `trace` 로 App Insights 에 감사 메타데이터 남기기 (Event Hub 불필요)

기존 Graph 정책을 **백업**한 뒤 inbound/outbound/on-error 에 `trace` 조각을 **주입**한다(게이트·MI 인증은 그대로 보존).
- inbound `trace` — 요청 시점에 **누가·무엇을·어떻게**(차단돼도 기록됨)
- outbound `trace` — **결과**(응답코드·백엔드 URL)
- on-error `trace` — 오류/차단 상세

`trace` 는 진단 verbosity(information) 이상이면 App Insights `traces` 로 방출된다(`Ocp-Apim-Trace` 헤더와 무관).

> ⚠️ **선행조건 2가지**(안 지키면 `traces` 0건): ① App Insights 진단 `verbosity=information` ② 설정 후 ~2분 **전파 대기** 뒤 트래픽. 셀 6 이 `ensure_ai_verbosity()`·`wait_propagation()` 로 **자동 처리**한다. 배경은 맨 끝 **트러블슈팅 타임라인** 참고.

In [ ]:
# 셀 6: Option A 적용 — (1) App Insights 진단 verbosity=information 보장  (2) 원 정책 백업+trace 주입  (3) 전파 대기
# ⚠️ 핵심①: 진단 verbosity 기본값은 사실상 error → severity="information" trace 가 전부 드롭된다(=traces 0건).
DIAG_PREEXISTED = ensure_ai_verbosity("information")

ORIG_POLICY = get_policy(GRAPH_API_ID)
assert "<inbound>" in ORIG_POLICY, "기존 Graph 정책을 읽지 못했습니다 (권한/ID 확인)."
print(f"  기존 정책 백업 완료 ({len(ORIG_POLICY)} chars)")

policy_A = inject(ORIG_POLICY, SETVARS + INBOUND_TRACE, OUTBOUND_TRACE, ONERROR_TRACE)
out, err, rc = put_policy(GRAPH_API_ID, policy_A)
print("✅ Option A 정책 적용 완료 (APIM 검증기 통과)" if rc == 0 else f"❌ 적용 실패: {err[:400]}")

# ⚠️ 핵심②: verbosity·정책 변경은 게이트웨이 전파에 ~1~2분. 지금 바로 트래픽을 쏘면 구설정으로 처리돼 누락된다.
wait_propagation(150)

In [ ]:
# 셀 7: Option A — 감사 트래픽 (허용 + select/filter + 차단)
uid = TEST_USER_ID
if KEY_USERS and "<" not in KEY_USERS:
    r = graph_get("/users", KEY_USERS, params={"$select":"id,displayName,mail","$top":"1"})
    if not uid and r.status_code == 200:
        try: uid = (r.json().get("value") or [{}])[0].get("id","")
        except Exception: uid = ""
    graph_get("/users", KEY_USERS, params={"$select":"displayName,mail,userPrincipalName","$filter":"accountEnabled eq true"})
    if uid:
        graph_get(f"/users/{uid}/messages", KEY_USERS, params={"$top":"3"})  # 403 기대 (정책 차단)
    print("  ✅ 트래픽 생성 (허용 + 차단)")
else:
    print("  ⚠️ graph-users 키 미설정 — 트래픽 생략")
print("  ⏳ App Insights 수집 2~5분 후 셀 8 실행")

In [ ]:
# 셀 8: Option A — App Insights 'traces' 감사 원장 확인 (재실행 가능)
print("▶ 정책이 남긴 감사 레코드 (누가·무엇을·어떻게·결과)")
show(query_ai("""
traces
| where timestamp > ago(30m)
| where message startswith "GRAPH-AUDIT"
| project timestamp,
    phase      = tostring(customDimensions.phase),
    product    = tostring(customDimensions.product),
    path       = tostring(customDimensions.path),
    targetUser = tostring(customDimensions.targetUser),
    ['select'] = tostring(customDimensions['select']),
    identity   = tostring(customDimensions.identity),
    keyHint    = tostring(customDimensions.keyHint),
    code       = tostring(customDimensions.responseCode)
| order by timestamp desc
| take 20
"""))
print("\n해석: BEFORE 엔 없던 targetUser·select·identity·keyHint 가 필드로 남음 = 행위 감사 성립.")
print("혹시 0건이면 → 맨 끝 '트러블슈팅 타임라인' 절차(verbosity·전파 대기)를 확인하세요.")

## Option B — Event Hub 로 전문(full body) 무손실 원장 남기기

Option A(메타데이터)로 부족한 경우 — **응답 전문·큰 페이로드**까지 남기려면 App Insights 크기 상한(수 KB)·샘플링에 걸린다.
`log-to-eventhub` 는 이 상한과 무관하게 이벤트(최대 ~1MB)를 Event Hub 로 보내고, **Event Hubs Capture** 가 이를 **Blob/ADLS 에 Avro 로 자동 아카이브**한다.

```
APIM (log-to-eventhub) ──▶ Event Hub ──(Capture)──▶ Blob/ADLS (불변 아카이브)
                                   └─(consumer group)─▶ SIEM · 실시간 이상탐지
```

> ⚠️ 아래 셀 9 는 **실제 리소스를 만들어 비용이 발생**한다(옵트인). `PROVISION_EVENTHUB=True` 로 바꿔야 실행된다.
> Event Hub 를 처음 쓴다면 셀 9 의 주석을 단계별로 읽어보자. 데모 자체는 Option A 만으로도 완결된다.

In [ ]:
# 셀 9(코드): Option B 프로비저닝 — Event Hub + Send/Listen 규칙 + APIM 로거  ⚠️ 비용·옵트인
PROVISION_EVENTHUB = False   # ← 실습하려면 True 로 변경

EH_NS = ""; EH_NAME = "graph-audit"; EH_CONN = ""
if not PROVISION_EVENTHUB:
    print("⏭️  PROVISION_EVENTHUB=False → Event Hub 생성 건너뜀. (Option A 만으로 감사 데모는 완결)")
    existing = arm_q("/loggers/graph-audit-eventhub", "name")
    print("   기존 APIM 로거 'graph-audit-eventhub':", "있음" if existing not in ("","null") else "없음")
else:
    import hashlib
    sfx = hashlib.sha1(RESOURCE_GROUP.encode()).hexdigest()[:8]
    EH_NS = f"ehnsgraphaudit{sfx}"
    LOC = az_json(["group","show","-n",RESOURCE_GROUP,"--query","location","-o","json"]) or "eastus2"
    print(f"▶ 1) Event Hub 네임스페이스 생성: {EH_NS} ({LOC}) — 1~2분")
    o,e,rc = az(["eventhubs","namespace","create","-g",RESOURCE_GROUP,"-n",EH_NS,"--sku","Standard","-l",LOC])
    print("   ", "ok" if rc==0 else e[:200])
    print(f"▶ 2) 이벤트 허브(토픽) 생성: {EH_NAME} (파티션 2)")
    az(["eventhubs","eventhub","create","-g",RESOURCE_GROUP,"--namespace-name",EH_NS,"-n",EH_NAME,"--partition-count","2"])
    print("▶ 3) 권한 규칙(Send=APIM, Listen=노트북) + 연결 문자열")
    az(["eventhubs","eventhub","authorization-rule","create","-g",RESOURCE_GROUP,"--namespace-name",EH_NS,
        "--eventhub-name",EH_NAME,"-n","apim-audit","--rights","Send","Listen"])
    EH_CONN = az_json(["eventhubs","eventhub","authorization-rule","keys","list","-g",RESOURCE_GROUP,
        "--namespace-name",EH_NS,"--eventhub-name",EH_NAME,"-n","apim-audit",
        "--query","primaryConnectionString","-o","json"]) or ""
    print("▶ 4) APIM 로거 'graph-audit-eventhub' 등록 (log-to-eventhub 가 참조)")
    o,e,rc = arm("PUT","/loggers/graph-audit-eventhub", body={"properties":{
        "loggerType":"azureEventHub","description":"Graph audit",
        "credentials":{"name":EH_NAME,"connectionString":EH_CONN}, "isBuffered":True}})
    print("   ✅ 로거 등록 완료" if rc==0 else f"   ❌ 로거 실패: {e[:200]}")
    print("✅ 프로비저닝 완료.")

# --- (선택) Capture → Blob 활성화 : 전문을 Blob 에 자동 아카이브 (리포트 §7 ②) -----------------
#   storageId=$(az storage account show -g <RG> -n <ST> --query id -o tsv)
#   az eventhubs eventhub update -g <RG> --namespace-name <NS> -n graph-audit \
#     --enable-capture true --capture-interval 300 --capture-size-limit 10485760 \
#     --destination-name EventHubArchive.AzureBlockBlob --storage-account $storageId --blob-container graph-audit
#   # + 네임스페이스 관리ID 에 'Storage Blob Data Contributor' 부여

## Option C — 진단설정(GatewayLogs) vs 정책(log-to-eventhub), 한 Event Hub 비교 · 준비(prime)

같은 `graph-audit` Event Hub에 **두 소스**를 흘려 차이를 본다: 진단설정(고정 GatewayLogs, **body 없음**) vs 정책(전문 body).
진단설정 로그는 전파에 수 분 걸리므로 여기서 **미리 켜두고**(prime) 소량 트래픽을 흘린다. Option B 이후 트래픽·대기 동안 전파돼 마지막 비교 셀에서 함께 보인다.

> ⚠️ 선행: 셀 9 `PROVISION_EVENTHUB=True` 로 `graph-audit` 허브가 있어야 한다. 진단설정은 허브 생성 **이후** 트래픽만 잡는다.


In [ ]:
# 셀 C1: [Option C 준비/prime] 진단설정(GatewayLogs)을 같은 graph-audit Event Hub 로 미리 연결
ENABLE_DIAG_TO_EVENTHUB = False  # ← True 로 바꿔야 실행 (⚠️ 비용·APIM 진단설정 변경)
if not ENABLE_DIAG_TO_EVENTHUB:
    print("⏭️  ENABLE_DIAG_TO_EVENTHUB=False → Option C(진단설정 비교) prime 생략.")
elif arm_q("/loggers/graph-audit-eventhub", "name") in ("", "null"):
    print("⏭️  graph-audit 로거 없음 → 셀 9 에서 PROVISION_EVENTHUB=True 로 먼저 생성하세요.")
else:
    ok, msg = enable_diag_to_eventhub()
    print("✅ 진단설정 'graph-audit-diag' 연결(prime)" if ok else f"❌ 진단설정 실패: {str(msg)[:200]}")
    if ok and KEY_USERS and "<" not in KEY_USERS:
        graph_get("/users", KEY_USERS, params={"$select": "displayName", "$top": "2"})
        print("  ✅ prime 트래픽 1회 → GatewayLogs 전파 타이머 시작(수 분 뒤 허브 도착)")


In [ ]:
# 셀 10(코드): Option B 정책 적용(= Option A + log-to-eventhub) 후 트래픽
logger_ok = arm_q("/loggers/graph-audit-eventhub", "name") not in ("", "null")
if not logger_ok:
    print("⏭️  로거 'graph-audit-eventhub' 없음 → Option B 정책 적용 생략.")
    print("   셀 9 에서 PROVISION_EVENTHUB=True 로 먼저 생성하세요 (로거 없으면 정책 검증 실패).")
else:
    # trace 메타데이터도 함께 보려면 A 와 동일하게 진단 verbosity 필요(멱등 호출)
    if "DIAG_PREEXISTED" not in dir():
        DIAG_PREEXISTED = ensure_ai_verbosity("information")
    else:
        ensure_ai_verbosity("information")
    base = ORIG_POLICY if ("ORIG_POLICY" in dir() and ORIG_POLICY) else get_policy(GRAPH_API_ID)
    if "ORIG_POLICY" not in dir() or not ORIG_POLICY:
        ORIG_POLICY = base; print("  (백업 없음 → 현재 정책을 백업으로 저장)")
    policy_B = inject(base, SETVARS + INBOUND_TRACE, OUTBOUND_TRACE + EH_OUT, ONERROR_TRACE + EH_ERR)
    out, err, rc = put_policy(GRAPH_API_ID, policy_B)
    print("✅ Option B 정책 적용 완료" if rc == 0 else f"❌ 적용 실패: {err[:400]}")
    if rc == 0:
        wait_propagation(150)   # ⚠️ 전파 대기 후 트래픽(안 그러면 trace/EH 둘 다 누락 위험)
        if KEY_USERS and "<" not in KEY_USERS:
            graph_get("/users", KEY_USERS, params={"$select":"displayName,mail","$top":"2"})
            if TEST_USER_ID:
                graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS, params={"$top":"2"})
            print("  ✅ 트래픽 생성 → Event Hub 로 전문 감사레코드 전송됨")

In [ ]:
# 셀 11(코드): Option B — 남은 감사 로그를 눈으로 (traces 즉시 + Event Hub 라이브 소비)
print("▶ (1) App Insights traces — Option B 도 trace 포함이라 메타데이터는 즉시 조회")
show(query_ai("""
traces | where timestamp > ago(20m) | where message startswith "GRAPH-AUDIT"
| project timestamp, phase=tostring(customDimensions.phase),
    targetUser=tostring(customDimensions.targetUser), identity=tostring(customDimensions.identity),
    code=tostring(customDimensions.responseCode)
| order by timestamp desc | take 10
"""))

print("\n▶ (2) Event Hub 라이브 소비 — 전문(full JSON) 레코드 직접 수신")
CONSUME_WITH_SDK = False   # ← True 로 바꾸면 azure-eventhub 로 실제 이벤트 수신(필요시 pip 설치)
if not CONSUME_WITH_SDK:
    print("   ⏭️  CONSUME_WITH_SDK=False. (셀 9 에서 EH_CONN 이 채워진 뒤 True 로 실행)")
elif not EH_CONN:
    print("   ⚠️ EH_CONN 없음 — 셀 9 에서 PROVISION_EVENTHUB=True 로 먼저 생성하세요.")
else:
    try:
        from azure.eventhub import EventHubConsumerClient
    except ImportError:
        print("   azure-eventhub 설치 중...")
        subprocess.run([sys.executable,"-m","pip","install","-q","azure-eventhub"])
        from azure.eventhub import EventHubConsumerClient
    import threading
    got = []
    client = EventHubConsumerClient.from_connection_string(EH_CONN, consumer_group="$Default", eventhub_name=EH_NAME)
    def on_event(ctx, ev):
        if ev is not None:
            got.append(ev.body_as_str())
            if len(got) >= 10:
                try: client.close()
                except Exception: pass
    def pump():
        try: client.receive(on_event, starting_position="-1", max_wait_time=5)
        except Exception: pass
    t = threading.Thread(target=pump, daemon=True); t.start()
    time.sleep(12)
    try: client.close()
    except Exception: pass
    t.join(timeout=3)
    print(f"   수신 {len(got)} 건:")
    for g in got[:10]:
        print("   •", g[:400])
    if not got:
        print("   (없음 — 트래픽 후 수 초~수십 초 지연. 셀 10 재실행 후 다시 시도)")

## Option C — 비교: 같은 허브, 두 소스의 이벤트 모양

한 `graph-audit` Event Hub 를 라이브 소비해 **정책(log-to-eventhub)** 이벤트와 **진단설정(GatewayLogs)** 이벤트를 봉투(shape) 기준으로 분류한다.
정책 이벤트는 전문 body 를 담고, 진단 이벤트는 고정 스키마라 body 가 없다 — **body 는 목적지가 아니라 소스(정책)가 만든다**.


In [ ]:
# 셀 C2: [Option C 비교] 정책 vs 진단설정 이벤트를 한 허브에서 분류·대조
CONSUME_C = False  # ← True 로 실행 (azure-eventhub 필요, EH_CONN 채워진 뒤)
if not CONSUME_C:
    print("⏭️  CONSUME_C=False. (셀 9 PROVISION_EVENTHUB=True 로 EH_CONN 채운 뒤 True)")
else:
    conn = EH_CONN if ("EH_CONN" in dir() and EH_CONN) else (az_json([
        "eventhubs","eventhub","authorization-rule","keys","list","-g",RESOURCE_GROUP,
        "--namespace-name",eh_ns_name(),"--eventhub-name",EH_NAME,"-n","apim-audit",
        "--query","primaryConnectionString","-o","json"]) or "")
    if not conn:
        print("  ⚠️ 연결문자열 없음 — 셀 9 에서 PROVISION_EVENTHUB=True 로 먼저 생성하세요.")
    else:
        try:
            from azure.eventhub import EventHubConsumerClient
        except ImportError:
            print("   azure-eventhub 설치 중...")
            subprocess.run([sys.executable,"-m","pip","install","-q","azure-eventhub"])
            from azure.eventhub import EventHubConsumerClient
        import threading
        got = []
        client = EventHubConsumerClient.from_connection_string(conn, consumer_group="$Default", eventhub_name=EH_NAME)
        def on_event(ctx, ev):
            if ev is not None:
                got.append(ev.body_as_str())
                if len(got) >= 50:
                    try: client.close()
                    except Exception: pass
        def pump():
            try: client.receive(on_event, starting_position="-1", max_wait_time=5)
            except Exception: pass
        t = threading.Thread(target=pump, daemon=True); t.start()
        time.sleep(20)
        try: client.close()
        except Exception: pass
        t.join(timeout=3)

        diag, policy = [], []
        for g in got:
            is_diag = False
            try:
                o = json.loads(g)
                recs = o.get("records") if isinstance(o, dict) else None
                if recs and isinstance(recs, list) and recs and recs[0].get("category") == "GatewayLogs":
                    is_diag = True
            except Exception:
                is_diag = False
            (diag if is_diag else policy).append(g)

        print(f"▶ 수신 {len(got)}건 → 정책 {len(policy)} / 진단(GatewayLogs) {len(diag)}")
        print("\n[정책 log-to-eventhub] 전문 body 포함:")
        for g in policy[:3]: print("  •", g[:300])
        print("\n[진단설정 GatewayLogs] 고정 스키마 · body 없음:")
        for g in diag[:3]: print("  •", g[:300])
        print("\n대조: 정책=body✅·커스텀필드✅ | 진단=body❌·고정필드 | 지연: 정책 초 vs 진단 수분")
        if not diag:
            print("  (진단 0건 — GatewayLogs 전파 지연. 수 분 뒤 이 셀 재실행)")


In [ ]:
# 셀 13: 원복 — 감사 조각 제거 + 임시 진단 정리, Lab 11 원래 정책으로 복원
if "ORIG_POLICY" in dir() and ORIG_POLICY:
    out, err, rc = put_policy(GRAPH_API_ID, ORIG_POLICY)
    print("✅ 원래 Graph 정책으로 복원 완료" if rc == 0 else f"❌ 복원 실패: {err[:300]}")
else:
    print("⚠️ 백업(ORIG_POLICY) 없음 — 셀 6(Option A 적용)을 먼저 실행했어야 합니다.")

# 우리가 새로 만든 API-레벨 App Insights 진단만 삭제(원래 있었으면 보존)
if "DIAG_PREEXISTED" in dir() and DIAG_PREEXISTED is False:
    _o, _e, rc = arm("DELETE", f"/apis/{GRAPH_API_ID}/diagnostics/applicationinsights")
    print("✅ 임시 API-레벨 진단 삭제(원복)" if rc == 0 else f"  (진단 삭제 스킵: {_e[:120]})")
print("  잔존 감사 trace?", "예(문제)" if "GRAPH-AUDIT" in (get_policy(GRAPH_API_ID) or "") else "아니오(clean)")

# (선택) Event Hub 리소스 정리 = 비용 중단:
#   az eventhubs namespace delete -g <RG> -n <EH_NS>
#   az rest --method DELETE --url "<ARM_BASE>/loggers/graph-audit-eventhub?api-version=2024-06-01-preview"

## 정리 — 언제 무엇을 쓰나

| 경로 | 얻는 것 | 잃는 것 | 언제 |
|---|---|---|---|
| **Option A** — `trace` → App Insights | 감사 메타데이터, 즉시·추가비용 0 | 전문✗(수 KB 상한·샘플링) | 행위 감사(누가/무엇/어떻게/결과)면 충분할 때 |
| **Option B** — `log-to-eventhub` → EH → **Capture → Blob** | 전문·커스텀 필드·팬아웃·불변 아카이브 | EH(+Capture) 비용 | 무손실 원장·장기·접근분리가 필요할 때 |
| (참고) 진단설정 → **Blob 직행** | 최저 비용·최단 구성 | 전문✗·커스텀필드✗·실시간✗ | 메타데이터만 Blob 에 싸게 보관 |

**핵심**
- 두 옵션 모두 **비동기 버퍼링** → 요청 레이턴시와 무관. Event Hub 는 *용량·무손실·불변* 때문에 쓴다.
- APIM 정책엔 `log-to-blob` 이 없다 → Blob 아카이브는 **Event Hubs Capture**(권장) 또는 `정책→Function` 경유.
- 결과: **하나의 APIM = 모든 솔루션의 Graph 접근 단일 통제·감사 지점.** "누가 어떤 사용자의 이메일을 언제 어떤 조건으로 가져갔는가"를 필드 단위로(A), 필요하면 전문·불변으로(B) 남긴다.

자세한 근거·필드 비교·KQL·공식 문서: [`docs/graph-gateway-audit-logging.md`](../../docs/graph-gateway-audit-logging.md)

## 🧭 트러블슈팅 타임라인 — "traces 0건" 을 만나면 (실측 기록)

> 이 노트북을 만들 때 **실제로 겪은 과정**이다. 증상은 *Option A 정책은 정상 적용(APIM 검증 통과)·트래픽도 200/403 정상인데 App Insights `traces` 가 계속 0건*. 같은 함정을 피하도록 **시간순**으로 남긴다.

| # | 시도한 것 | 결과 | 배운 것 |
|---|---|---|---|
| 1 | verbosity 손 안 대고 정책만 적용 → **즉시** 트래픽 → 조회(6분 재시도) | ❌ 0건 | 정책·트래픽은 정상인데 trace 만 안 남음 |
| 2 | 로거/진단 점검 (`/loggers`, `/diagnostics`) | 🔎 | App Insights 로거·서비스 진단은 있으나 **`verbosity`=null(=error 기본)**, graph API-레벨 진단 없음 |
| 3 | App Insights sanity: `union * \| summarize count() by itemType` | 🔎 | 최근 `request` 는 수집되는데 **`trace` 는 7일간 0건** = trace 정책이 여태 한 번도 방출된 적 없음 |
| 4 | **API-레벨** 진단 verbosity=information → *대기 없이* 트래픽 | ❌ 0건 | verbosity 만으로는 부족? → 다음 의심 |
| 5 | 공식 문서 확인 | 📖 | 조건은 오직 *severity ≥ verbosity*; **`Ocp-Apim-Trace` 헤더·구독 트레이싱은 이제 미지원** |
| 6 | **서비스-레벨** verbosity=information (GET 로 확인) → *대기 없이* 트래픽 | ❌ 0건 | 설정은 맞는데도 0 → **전파 지연** 의심 |
| 7 | verbosity=information → **150초 대기** → 트래픽 | ✅ **12건** | inbound/outbound·200/403·targetUser·identity·keyHint 전부! |
| 8 | **API-레벨** verbosity + **150초 대기** (전역 미변경) | ✅ **12건** | 전역 안 건드리는 국소 방식으로 재현 → **이 노트북이 채택한 방식** |

### 결론 — 두 조건을 **동시에** 만족해야 trace 가 App Insights 에 남는다
1. **App Insights 진단 `verbosity` ≥ `information`** — 기본값은 사실상 `error` 라 `severity="information"` 트레이스를 전부 버린다. → 셀 6 의 `ensure_ai_verbosity()` 가 graph **API-레벨** 진단에만 적용(다른 API 영향 0).
2. **설정 후 게이트웨이 전파(~1~2분)를 기다린 뒤 트래픽** — verbosity·정책 PUT 직후 곧바로 호출하면 *구 설정* 으로 처리돼 누락된다. → 셀 6 의 `wait_propagation(150)`.

### ✅ 사용자 체크리스트 (같은 실수 방지)
- [ ] **진단 verbosity 를 `information`(또는 `verbose`) 로.** `error`(기본)면 감사 trace 가 안 남는다.
- [ ] verbosity·정책 **변경 후 최소 2분 대기 → 그 다음 트래픽.** (즉시 호출 ✗)
- [ ] 조회는 **App Insights 수집 2~5분 지연** 이 정상 → 셀 8 을 몇 번 재실행.
- [ ] `Ocp-Apim-Trace` 헤더로 켜려 하지 말 것 — **폐기됨**. 진단 verbosity 가 유일한 스위치다.
- [ ] 가능하면 **서비스(전역) 대신 API-레벨 진단** 으로 국소 적용(다른 API 로그량·비용 영향 차단).
- [ ] 빠른 진단: `traces | summarize count() by itemType` 가 0 이면 verbosity/전파 문제. `union * | summarize count() by itemType` 로 *request 는 있는데 trace 만 0* 이면 확진.
- [ ] `trace` 는 **App Insights 샘플링 영향 없음**(공식 문서) → 샘플링은 원인이 아니다.

> Option B(Event Hub) 는 verbosity 와 무관하지만(App Insights 경로가 아님), 위 **전파 대기**는 동일하게 적용된다 — 정책 PUT 직후 트래픽은 log-to-eventhub 도 누락될 수 있다.
